XLA Notes:
https://openxla.org/xla/architecture

The XLA compiler takes model graphs from ML frameworks defined in StableHLO and compiles them into machine instructions for various architectures. StableHLO defines a versioned operation set (HLO = high level operations) that provides a portability layer between ML frameworks and the compiler.

In general, the compilation process that converts the model graph into a target-optimized executable includes these steps:

XLA performs several built-in optimization and analysis passes on the StableHLO graph that are target-independent, such as CSE, target-independent operation fusion, and buffer analysis for allocating runtime memory for the computation. During this optimization stage, XLA also converts the StableHLO dialect into an internal HLO dialect.

XLA sends the HLO computation to a backend for further HLO-level optimizations, this time with target-specific information and needs in mind. For example, the GPU backend may perform operation fusions that are beneficial specifically for the GPU programming model and determine how to partition the computation into streams. At this stage, backends may also pattern-match certain operations or combinations thereof to optimized library calls.

The backend then performs target-specific code generation. The CPU and GPU backends included with XLA use LLVM for low-level IR, optimization, and code generation. These backends emit the LLVM IR necessary to represent the HLO computation in an efficient manner, and then invoke LLVM to emit native code from this LLVM IR.

Within this process, the XLA compiler is modular in the sense that it is easy to slot in an alternative backend to target some novel HW architecture. The GPU backend currently supports NVIDIA GPUs via the LLVM NVPTX backend. The CPU backend supports multiple CPU ISAs.

In [ ]:
import os, shutil, pathlib

dump_dir = pathlib.Path("/tmp/xla")
if dump_dir.exists(): shutil.rmtree(dump_dir)
dump_dir.mkdir(parents=True, exist_ok=True)

os.environ["XLA_FLAGS"] = (
    f"--xla_dump_to={dump_dir} "
    "--xla_dump_hlo_pass_re=.* "
    "--xla_dump_hlo_as_text "
    "--xla_dump_module_metadata_to_file=false "
    "--xla_cpu_enable_fast_math=true"
)
print("Dump dir:", dump_dir)

In [ ]:
#make sure you do this after setting os.environ flags
import tensorflow as tf, numpy as np, sys
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("TensorFlow version:", tf.__version__)
print("Available devices:", tf.config.list_physical_devices())

### LOAD A TRAINED MODEL, OBSERVE, OPTIMIZE USING XLA

In [ ]:
# 1A. Get pretrained ResNet50
model = tf.keras.applications.MobileNetV2(weights=None, input_shape=(224,224,3))
model.summary()

# 1B. Save as a SavedModel (to simulate using an existing one)
save_path = "/tmp/resnet50_saved.keras"
model.save(save_path)
print("Model saved to:", save_path)

# 1C. Load back from disk
model = tf.keras.models.load_model(save_path)

In [ ]:
#This wraps the Keras model inside a TensorFlow function that is compiled with XLA (Accelerated Linear Algebra).
#Unlike eager execution, @tf.function(jit_compile=True) does lazy staging — TensorFlow traces the Python function, 
#builds a computation graph, and then compiles it into an XLA executable the first time you call it.
#Note with this call @tf.functiom Tensorflow:::
#Traces the Python function f the first time it’s called with a particular input signature.
#Builds a TensorFlow graph (a GraphDef) from that trace.
#Creates a callable ConcreteFunction — a compiled TensorFlow graph that runs in C++, bypassing Python overhead.
#That’s already a “graph,” but it’s still a TensorFlow-level graph, not an XLA-compiled binary yet.

#when you add jit_compile=True:::
#Builds a TF graph, then hands it to XLA
#Lowers to HLO IR (XLA’s format)
#Optimized by XLA compiler passes
#Runs with XLA-generated kernels
#So @tf.function gives you a TensorFlow graph; adding jit_compile=True gives you an XLA-compiled program.
@tf.function(jit_compile=True)
def run_resnet(x):
    return model(x)

In [ ]:
#1. Tracing
# TensorFlow records the operations (Conv2D, MatMul, BatchNorm, etc.) that make up your ResNet model, 
#using the concrete shapes and dtypes of x (here, (1, 224, 224, 3)).

#2. Graph building
#Those ops are converted into an internal computation graph (a “tf.Graph”).

#3. XLA compilation
#Because jit_compile=True, TensorFlow passes the graph to the XLA compiler, which lowers it to HLO (High-Level Optimizer) IR.
#That triggers all the optimization passes (fusion, constant folding, algebraic simplification, etc.), and it’s 
#/ where /tmp/xla_dump_tf files are generated.

#4. Executable creation
#XLA produces a platform-specific executable (CPU in this case) that it caches so that subsequent calls reuse the same
#compiled binary.

#5. Execution
#Finally, the compiled code runs once with your sample data and returns an output tensor — but we ignore it (_ = …)
#because the goal is to trigger compilation, not training or inference yet.

#run_resnet(x) “materializes” the compiled function for that input signature.experimental_get_compiler_ir(x) then queries that compiled artifact to show its internal IR.
import numpy as np
x = tf.constant(np.random.randn(1, 224, 224, 3), dtype=tf.float32)
_ = run_resnet(x)  # this is warm-up compilation--> The first call to a compiled function triggers tracing + compilation + execution 
#(expensive). Later calls reuse the compiled graph (fast).

In [ ]:

#This is the initial HLO graph generated from the TensorFlow ops, roughly a one-to-one mapping of TF ops -->
#HLO instructions. It lowers TensorFlow ops (like Conv2D, BatchNorm, MatMul, Relu) into HLO instructions.
#Each HLO is a pure, functional operation with well-defined semantics — like add, dot_general, reduce, fusion, etc.
# At this stage: The HLO graph mirrors your model structure. It’s still high-level: every TF op is mostly a distinct HLO op.
# Nothing has been fused or simplified yet.

hlo_text = run_resnet.experimental_get_compiler_ir(x)(stage="hlo")

#This calls the compiler again, but now asks for the IR after all XLA optimization passes 
#(fusion, constant folding, simplification, etc.). This shows what the compiler will actually execute — 
#often with many fewer ops, fusions, or reordered computations. Each pass either: removes redundancy, fuses operations, or 
#reorders computations to improve locality and performance.
#Some Optimizations Performed are:
#Algebraic Simplifier	x * 1 → x, exp(log(x)) → x, 	Remove redundant math
#Constant Folding	x + 3 + 4 → x + 7	Compute at compile time
#CSE	Duplicate x*y-->z1 = x * y;z2 = x * y + 3	Compute once & reuse
#Fusion	relu(x+b)	It merges multiple elementwise ops into a single kernel so that data stays in registers instead of writing intermediate results to memory.
                #XLA has several fusion strategies: #Loop fusion: merge simple elementwise ops; Input fusion: push computations into producer ops; Output fusion: merge consumer ops into producer.
#Layout Assignment	Memory order	Improve data locality--> XLA decides how to lay out tensors in memory (row-major vs column-major, alignment, padding).It also reorders operations to minimize transposes or reshapes.
#Copy Elimination	transpose(transpose(x))	Remove redundant ops--> Removes unnecessary reshapes, transposes, or copies introduced during earlier lowering.Also merges adjacent reshapes if they cancel out.
#Dead Code Elimination	Unused nodes	Shrink graph,Removes computations whose results are never used (often happens after CSE or fusion). 
#Dot Optimization	matmul tuning	Use best library kernels
#HLO Pass Cleanup / Canonicalization --> After all major passes, XLA performs a final canonicalization pass that reorders and names ops for deterministic output and removes stray tuples or trivial ops.
#Buffer Assignment	Memory reuse	Reduce allocation overhead, XLA plans memory buffers — reusing them when safe (like SSA register allocation). This is how it avoids allocating new memory for every temporary.

opt_hlo_text = run_resnet.experimental_get_compiler_ir(x)(stage="optimized_hlo")


#Converts each IR object to a string and prints just the first ~800 characters (because the full HLO can be 
#thousands of lines long). The output shows you side by side:
#Before optimization: more explicit, many small element-wise ops.
#After optimization: fused kernels, simplified arithmetic, folded constants, etc.
print("=== HLO (pre-opt) ===\n", str(hlo_text)[:800])
print("\n=== Optimized HLO ===\n", str(opt_hlo_text)[:800])

In [ ]:
## ANALYZE IMPACT OF OPTIMIZATIONS

hlo_raw_text = str(hlo_raw)
hlo_opt_text = str(hlo_opt)

#Compute unified diff
import difflib
diff = list(difflib.unified_diff(
    hlo_raw_text.splitlines(),
    hlo_opt_text.splitlines(),
    lineterm="",
    n=3
))
print("\n".join(diff[:200])) #lines starting with - --> removed (usually eliminated ops) lines starting with + --> added (usually fused ops or simplified constants)

## Quantify what changed
import re, collections
def count_ops(hlo):
    return collections.Counter(re.findall(r"=\s*(\w+)", hlo))

raw_counts = count_ops(hlo_raw_text)
opt_counts = count_ops(hlo_opt_text)

print(f"{'Op':20} {'Raw':>5} {'Opt':>5} {'Δ':>4}")
for op in sorted(set(raw_counts)|set(opt_counts)):
    r, o = raw_counts.get(op,0), opt_counts.get(op,0)
    if r!=o:
        print(f"{op:20} {r:5} {o:5} {o-r:+4}")

## Highlight & annotate optimizations
from IPython.display import Markdown
summary = []

if "fusion" in hlo_opt_text:
    summary.append("✅ **Fusion:** many small elementwise ops merged into fused kernels.")
if len(hlo_opt_text) < len(hlo_raw_text):
    summary.append("✅ **Dead code & simplification:** total HLO text shrank.")
if "broadcast" in hlo_raw_text and "broadcast" not in hlo_opt_text:
    summary.append("✅ **Broadcast elimination:** fused away redundant broadcasts.")
if "copy" in hlo_raw_text and "copy" not in hlo_opt_text:
    summary.append("✅ **Copy elimination:** removed intermediate tensor copies.")
if "dot" in hlo_raw_text and hlo_raw_text.count("dot") != hlo_opt_text.count("dot"):
    summary.append("✅ **Dot optimization:** merged bias/activation into GEMM.")

Markdown("### 🔍 Optimization summary\n" + "\n".join(summary))


In [ ]:
import glob
files = sorted(glob.glob(f"{dump_dir}/**/*.txt", recursive=True))
print(f"Found {len(files)} dump files")
print("\n".join(files[:10]))


In [ ]:
# This block of code is how you measure and compare performance between normal TensorFlow execution (eager mode)
#and XLA-compiled execution.

import time

def time_run(fn, x, iters=10):
    _ = fn(x)  # warm-up XLA compilation occurs lazily, on the first call with a given input shape. That call can take seconds because XLA runs all optimization passes and code-generates the kernel.Subsequent calls reuse the compiled binary and are much faster.The time_run() function skips that compilation time so you measure steady-state inference performance.
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = fn(x).numpy()
    t1 = time.perf_counter()
    return (t1 - t0) / iters

# Eager (no XLA) it executes Python ops one-by-one (fast but not compiled).Operations are executed immediately, one at a time, via the TensorFlow runtime.
# No graph or compiler optimizations — each op is a separate kernel call.
def eager_infer(x): return model(x)

eager_time = time_run(eager_infer, x)
#run_resnet is the XLA-compiled function (@tf.function(jit_compile=True)). This path uses the XLA compiler to run a single optimized binary that fuses many ops together.
xla_time = time_run(run_resnet, x)

print({
    "eager_s/iter": eager_time,
    "xla_s/iter": xla_time,
    "speedup_XLA_vs_eager": eager_time / xla_time
})

In [ ]:
#By adding one simple op (y + 1.0 → relu), you create a modified computation whose HLO you can compare to the original.
#This lets you see XLA’s optimizations in action: Before: lots of small ops in separate lines. After: a fusion node with sub-computation body containing your new ops.

@tf.function(jit_compile=True)
def run_resnet_opt(x):
    y = model(x)
    # Extra elementwise ops that XLA can fuse
    return tf.nn.relu(y + 1.0)

_ = run_resnet_opt(x)
opt2_hlo = run_resnet_opt.experimental_get_compiler_ir(x)(stage="optimized_hlo")
print(str(opt2_hlo)[:800])

In [ ]:
with open("/tmp/resnet_optimized_hlo.txt", "w") as f:
    f.write(str(opt_hlo_text))

In [ ]:
import difflib

def diff_hlos(hlo1: str, hlo2: str, context=3):
    hlo1_lines = str(hlo1).splitlines()
    hlo2_lines = str(hlo2).splitlines()
    diff = difflib.unified_diff(hlo1_lines, hlo2_lines, lineterm="", n=context)
    return "\n".join(diff)

diff_text = diff_hlos(opt_hlo_text, opt2_hlo)
print(diff_text[:2000])

In [ ]:
import re

def extract_ops(hlo_txt):
    return [m.group(1) for m in re.finditer(r"^%([\w\d_.-]+)", str(hlo_txt), re.MULTILINE)]

ops1 = extract_ops(opt_hlo_text)
ops2 = extract_ops(opt2_hlo)

removed = sorted(set(ops1) - set(ops2))
added = sorted(set(ops2) - set(ops1))

print("Removed ops:", removed[:10])
print("Added ops:", added[:10])
print("Δ total ops:", len(ops2) - len(ops1))

### CREATE OUR OWN MODEL AND OPTIMIZE USING XLA

In [ ]:
from tensorflow.keras import layers, models

def create_model():
    model = models.Sequential([
        layers.Dense(64, activation='relu', input_shape=(784,)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model_1 = create_model()
model_1.summary()

In [ ]:
@tf.function(jit_compile=True)
def train_step(model_1, x, y):
    with tf.GradientTape() as tape:
        preds = model_1(x, training=True)
        loss = tf.keras.losses.sparse_categorical_crossentropy(y, preds)
    grads = tape.gradient(loss, model_1.trainable_variables)
    model_1.optimizer.apply_gradients(zip(grads, model_1.trainable_variables))
    return loss

import numpy as np

x = np.random.rand(256, 784).astype(np.float32)
y = np.random.randint(0, 10, size=(256,))

for step in range(100):
    loss = train_step(model_1, x, y)
print("Training complete!")

Next we can repeat above steps on this..